In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms

root_path = '.'

In [ ]:
trans_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

trans_baseline = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

trans_basico = transforms.Compose([
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

trans_avancado = transforms.Compose([
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    transforms.RandomErasing()
])

In [ ]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.fc1 = nn.Sequential(
            nn.Linear(7 * 7 * 64, 1000),
            nn.ReLU())
        self.fc2 = nn.Linear(1000, 10)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.reshape(out.size(0), -1)
        out = self.fc1(out)
        out = self.fc2(out)
        return out
model = ConvNet()
print(model)

In [ ]:
batch_size = 32
learning_rate = 0.001
epochs = 6
loss_fn = nn.CrossEntropyLoss()

test_dataset = torchvision.datasets.MNIST(root=root_path, train=False, transform=trans_test, download=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
def treinar(nome, train_loader, mixup=False):
    model = ConvNet()
    optimizer = Adam(model.parameters(), lr=learning_rate)
    loss_list = []
    acc_list = []
    total_step = len(train_loader)

    print(f"\n{'='*55}")
    print(f"  Pipeline: {nome}")
    print(f"{'='*55}")

    for epoch in range(epochs):
        for i, (images, labels) in enumerate(train_loader):

            # ── Mixup ────────────────────────────────────────────────────────
            if mixup:
                lam = torch.distributions.Beta(0.4, 0.4).sample()
                idx = torch.randperm(images.size(0))
                images = lam * images + (1 - lam) * images[idx]
                labels_a, labels_b = labels, labels[idx]
                outputs = model(images)
                loss = lam * loss_fn(outputs, labels_a) + (1 - lam) * loss_fn(outputs, labels_b)
            else:
                outputs = model(images)
                loss = loss_fn(outputs, labels)
            # ─────────────────────────────────────────────────────────────────

            loss_list.append(loss.item())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total = labels.size(0)
            _, predicted = torch.max(outputs.data, 1)
            correct = (predicted == labels).sum().item()
            acc_list.append(correct / total)

            if (i + 1) % 100 == 0:
                print(f"Época [{epoch+1}/{epochs}], Step [{i+1}/{total_step}], "
                      f"Custo: {round(loss.item(), 3)}, Acurácia: {round((correct / total) * 100, 3)}")
            model.eval()
            with torch.no_grad():
                correct = 0
                total = 0
                for images, labels in test_loader:
                    outputs = model(images)
                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()
                acc_final = round((correct / total) * 100, 3)
                print(f"\n>>> Acurácia final no teste [{nome}]: {acc_final}%")

            return loss_list, acc_list, acc_final


In [ ]:
pipelines = [
    ("Baseline",  trans_baseline, False),
    ("Basico",    trans_basico,   False),
    ("Avancado",  trans_avancado, False),
    ("Mixup",     trans_baseline, True),
]

resultados = {}

for nome, trans, mixup in pipelines:
    train_dataset = torchvision.datasets.MNIST(root=root_path, train=True, transform=trans, download=True)
    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    loss_list, acc_list, acc_final = treinar(nome, train_loader, mixup=mixup)
    resultados[nome] = {"loss": loss_list, "acc": acc_list, "acc_final": acc_final}


In [ ]:
print(f"\n{'='*55}")
print("  RESUMO FINAL")
print(f"{'='*55}")
for nome, res in resultados.items():
    print(f"{nome:10s} → Acurácia no teste: {res['acc_final']}%")